## **Querying notebook**

Connect to the same database:

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

Check how many documents are in the index:

In [2]:
sqlite_index.count()

84

Try a search:

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

## **RAG with sqlitesearch**    

We use the RAGBase class from rag_helper.py with this sqlitesearch index.   

Because our RAG is modular, we just swap the search index - the rest of the code stays the same:

In [4]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

This code skips both the fit call and the data loading. The index is already populated by the ingestion notebook, so we just connect to the database file.

Try it:

In [5]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it started. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


The data comes from a persistent index - no fetching, no processing, no indexing at startup. And we didn't have to rewrite any of the RAG logic - just swapped the index.

The modular design splits the work cleanly:

ingest.py handles data loading and indexing
rag_helper.py handles the RAG pipeline
the notebooks wire them together
This works because sqlitesearch follows the same API as minsearch - both have a search method that takes a query, boost_dict, filter_dict, and num_results. If the API were different, we'd need to subclass RAGBase and override the search method to adapt to the new backend.

## **Comparing the two approaches**
With minsearch (single process):

Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
With sqlitesearch (two processes):

Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready

The full architecture:

![alt text](RAG_full_architecture_ingestion.png)

The ingestion process writes documents to the knowledge base.

The RAG assistant then reads from it:

![alt text](RAG_full_architecture_querying.png)

For our FAQ dataset, both produce good results. The difference matters more at scale with diverse document lengths.

## **Choosing an approach**
Pick the right tool for your data:

minsearch: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
sqlitesearch: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.
Use minsearch when you can load and index the data on startup without noticeable delay. Switch to a persistent backend when ingestion takes too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different backend:

Elasticsearch
OpenSearch
Qdrant (vector database)
Weaviate (vector database)
The architecture stays the same: one process ingests, another queries.

## **Cleaning up**
When you're done, close the database connection:

sqlite_index.close()
Or just let Python clean it up when the notebook kernel shuts down.